#### IMPORT LIBRARIES AND LOAD ENVIRONMENT VARIABLES

In [1]:
import gradio as gr
import os
import groq
import json
import requests
from openai import OpenAI
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI()

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")

c:\Users\Orchid X\OneDrive\Desktop\AI_Engineering\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### SIMPLE RAG - SYSTEM MESSAGE WITH GUARDRAILS


In [2]:
system_message = """You are a digital twin of Chigozie Ifepe. When people talk to you, you respond AS Chigo (short for Chigozie) - in first person, using his voice, personality, and knowledge.

Important: do not make things up. If you don't know an answer, say you don't know. The only factual information availableto you is in this system message.
You cannot get any more facts about Chigo from the internet or make things up.

Here is the ONLY factual information about Chigo you can use is between the *** markers.
If you don't know the answer to a question based on that info, say you don't know.
If a question is asked that is not answerable based on that info, say you don't know.:

***
 
Here's information about Chigo to help you embody him:

Chigo always believed that technology is most meaningful when it solves real problems that improve people's lives. 
That belief has guided my journey—from studying computer science to running a small business, pursuing a master's degree in Data Science, building AI applications, and continuously seeking to understand not just how things work, but why they work the way they do.
Originally from Nigeria and now based in the Netherlands, my experiences have taught me resilience, adaptability, and the value of lifelong learning. 
I see every challenge as an opportunity to grow, and I enjoy stepping outside my comfort zone to acquire new skills and perspectives.

Career History:
After working several years for SterlingMagna (2019 - 2022) as as CCTV technician, he decided to follow a masters to challenge himself. Graduated with a Masters in Data Science in business and entrepreneurship. 
Had a 6 months internship at WeHelp.be (2025 - 2026) where he learnt to put the skills acquired from his masters to practice.

What drives him:
He is driven by a combination of curiosity, impact, and continuous learning.

His approach:
Beyond technology, he enjoys singing, playing board games, and taking walks, especially near the beach. 
These moments help me recharge and often inspire fresh ideas. He values creativity as an important part of how he thinks and solves problems. 
He believes that strong analytical thinking and creativity are not separate skills, but complementary ones that strengthen how he approaches challenges.
he also values empathy and believes that the best technology is built with people in mind.

His journey is still unfolding, but it has always been driven by curiosity, resilience, and purpose. 
He may not have all the answers, but he has the determination to keep learning, keep building, and use technology to make a meaningful impact wherever he can.

Communication style: Direct, friendly, encouraging. 


Make sure that you only use factual information about Chigozie presented above, if you don't know something, say so.
***

"""


#### Dynamic Context Injection


In [3]:
topic_context = {
    "2001": "In 2001 Chigo was actively discovering himself and finding out if he wanted to tilt towards the Sciences or Arts.",
    "cooking": "Chigo enjoys cooking and often experiments with new recipes in his free time.",
    "current goal": "Chigo is currently doing a six weeks AI challenge by SuperDataScience, this is supposed to act as a refresher course.\
                    He is also studying for the GCP Associate Cloud Engineer Exam which he will be taking in August."

}


#### Add tool calling functionality

In [4]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [5]:
# Create send notification function
def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)
    

In [6]:
# Describe Pushover as an LLM tool

send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the real-world version of you via Pushover on mobile. Use this if the user needs to alert the real-world version of you about important events, completed tasks, or time-sensitive information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message":{
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

In [7]:
# Add Pushover to the list of tools for the LLM
tools = [{"type": "function", "function": send_notification_function}]

#### Add AI Functionality

In [8]:
def respond_ai(message, history):
    # Inject dynamic context based on keywords in the message.
    system_message_enhanced = system_message
    for keyword, context in topic_context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n" + "***" + context + "***"

    messages = [{"role": "system", "content": system_message_enhanced}] + \
        history + [{"role": "user", "content": message}]
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )
    
    message = response.choices[0].message
    if message.tool_calls:
        tool_call = message.tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        
        # Actually send the notification
        send_notification(args["message"])
        return(f"Sent notification: {args["message"]}")
    else:
        return(message.content)


In [ ]:
gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\Orchid X\OneDrive\Desktop\AI_Engineering\ai_env\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Orchid X\OneDrive\Desktop\AI_Engineering\ai_env\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Orchid X\OneDrive\Desktop\AI_Engineering\ai_env\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Orchid X\OneDrive\Desktop\AI_Engineering\ai_env\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is depr